![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 07: Model Adaptation and Multimodal GenAI)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 7E: Fast Inference Provider Comparison

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local workflow that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Optional live API timing if keys are available.</td></tr>
<tr><td align="left">Main output</td><td>Compare inference providers using a local scoring matrix.</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m07e-overview)
2. [Conceptual Background](#m07e-background)
3. [Setup](#m07e-setup)
4. [Approved Local Data](#m07e-data)
5. [Mandatory Local Workflow](#m07e-workflow)
6. [Inspection and Interpretation](#m07e-inspection)
7. [Optional Real Model or Package Section](#m07e-optional)
8. [Testing and Analysis](#m07e-testing)
9. [Student Tasks](#m07e-tasks)
10. [Submission and Reflection](#m07e-submission)

---

<a id="m07e-overview"></a>

### 1. Overview and Learning Goals

This session is **M07E: Fast Inference Provider Comparison**. Once a model is trained or adapted, someone still has to serve it — and different inference providers make very different trade-offs between speed, cost, throughput, privacy, and quality. In this session you build a local, evidence-based comparison workflow: the kind of scoring matrix an engineering team assembles before committing to a provider, constructed here without needing any live API keys.

The central theme is:

```text
Compare inference providers using a local scoring matrix.
```

The session deliberately starts with a mandatory local workflow that does not need an API key, a paid model endpoint, or a live external service. This matters for two reasons. First, everyone can complete the core learning even without credits or a GPU. Second, the main learning objective is the architecture — how information is represented, processed, checked, and turned into a safe output — and a small local simulation makes that architecture fully visible. Once you can see and test every step here, the same skeleton carries over directly to the real tools discussed in the optional section.

The concepts used in this session are:

```text
1. latency
2. cost
3. throughput
4. privacy
5. quality
6. provider selection
```

Every run of the workflow follows the same general shape:

```text
User request
     |
     v
Validate input ............ refuse empty or unsafe requests early
     |
     v
Select approved context .... pick only relevant, approved local items
     |
     v
Apply workflow logic ....... transform the request using that context
     |
     v
Structured output .......... status + summary + evidence + limitations
     |
     v
Inspect and test ........... check grounding, refusals, failure cases
```

By the end of this session, you should be able to describe this workflow in your own words, run the mandatory local implementation, inspect its intermediate outputs, add a small extension of your own, test normal, edge, and failure cases, and explain how the design would change if a real model or external package were added.

<a id="m07e-background"></a>

### 2. Conceptual Background

The goal of this practical is not just to make a notebook run. The goal is to understand the design discipline behind an agentic AI workflow, and then to see how that discipline applies to inference-provider comparison.

A weak workflow often looks like this:

```text
User request ---> one large prompt ---> model output
```

This is simple, but it hides too many decisions. You cannot tell whether the input was valid, whether the right context was used, whether the output was safe, or whether the system should have refused or asked for clarification. When something goes wrong, there is no seam to open and inspect.

A stronger workflow separates the steps so each one can be checked on its own:

```text
User request
     |
     v
Input validation .......... was the request well-formed and allowed?
     |
     v
Context selection ......... which approved evidence supports an answer?
     |
     v
Controlled transformation . logic you can read, not a black box
     |
     v
Structured output ......... status, summary, evidence, limitations
     |
     v
Tests and review .......... normal, edge and failure cases all covered
```

For **Fast Inference Provider Comparison**, the session's six concepts play these roles:

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>What it means in practice</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">latency</td><td>How long a user waits. Two numbers matter: time to first token (which dominates perceived speed in chat) and total completion time.</td></tr>
<tr><td align="left">cost</td><td>Price per million input and output tokens; output tokens usually cost several times more. Prices change often, so every quote needs a date.</td></tr>
<tr><td align="left">throughput</td><td>Tokens generated per second. It matters most for batch jobs and long outputs, and can be high even when first-token latency is poor.</td></tr>
<tr><td align="left">privacy</td><td>Where requests are processed, how long they are retained, and whether they can be used for training. Often the deciding factor for organisations, whatever the speed numbers say.</td></tr>
<tr><td align="left">quality</td><td>Task-specific output quality. The fastest, cheapest provider is not automatically good enough for your task — quality needs its own evaluation.</td></tr>
<tr><td align="left">provider selection</td><td>A weighted trade-off across all of the above. Different use cases — interactive chatbot versus overnight batch pipeline — legitimately pick different winners.</td></tr>
</tbody>
</table>

</div>

A beginner-friendly analogy: choosing an inference provider is like choosing a courier for a parcel. The overnight courier is fast and expensive, the postal service is cheap and slow, and the specialist handler is slower still but insured and confidential. There is no universally best courier — it depends on what you are shipping and to whom. A scoring matrix makes that dependence explicit instead of hiding it inside a gut feeling.

```text
Fixed prompt set (same prompts, same max_tokens, versioned)
        |
        +--> Provider 1 --> warm-up call, then >= 3 timed runs
        |                   record: first-token latency, total time,
        |                           tokens/sec, cost, quality notes
        +--> Provider 2 --> (same procedure)
        |
        +--> Provider 3 --> (same procedure)
        |
        v
Take medians --> normalise scores --> apply use-case weights
        |
        v
Weighted scoring matrix --> ranked recommendation + stated caveats
```

Method choices that make results trustworthy: send one warm-up request per provider and exclude it (a cold start can double a latency figure), repeat each measurement at least 3 times and report the median rather than the mean, and hold the prompt set and `max_tokens` identical across providers. Failure behaviours to recognise: HTTP 429 responses mean you hit a rate limit (space out calls and retry with backoff), and wildly inconsistent numbers usually mean a missing warm-up or mismatched output lengths rather than a genuinely erratic provider.

The mandatory workflow in this notebook uses a local simulation because local simulations make the control structure visible. Real models and packages can be added later — the optional section shows how — but they should never replace validation, inspection, tests, stated limitations, and human review.

<a id="m07e-setup"></a>

### 3. Setup

The mandatory part of this notebook uses only the Python standard library. It runs on a plain CPU runtime in Google Colab or local Jupyter — you do not need a GPU, an API key, or any package installation for the core work. The optional live-timing section needs provider API keys (loaded with `getpass`, never pasted into a cell) but no GPU — a normal CPU runtime is enough for this entire notebook.

The safety boundary for this session is:

```text
1. Use only approved public-style or synthetic teaching data.
2. Do not use private documents, credentials, emails, student records or hidden instructor materials.
3. Do not perform real external side effects.
4. Show limitations when the local workflow does not have enough information.
5. Keep output inspectable and testable.
```

If you attempt the optional section later, keep any credential in an environment variable loaded with `getpass` — never paste a key into a code cell.

In [ ]:
# The mandatory workflow deliberately uses only the Python standard library,
# so it runs identically on a free Colab CPU runtime or on a local machine.
# Design decision: zero external packages means zero version conflicts and
# zero hidden behaviour -- every step you observe is code you can read here.

import json  # pretty-printing structured results so they are easy to inspect
import re    # lightweight tokenisation for the relevance-matching step
from typing import Any, Dict, List

print("Setup complete.")

<a id="m07e-data"></a>

### 4. Approved Local Data

The local data below is synthetic teaching data created for this practical. It contains nothing private, and it is deliberately tiny — three items — so that you can read every record and predict, before running anything, why the workflow will or will not select it. That predictability is the point: when a workflow misbehaves, you debug it by comparing what it did against what the data says it should have done.

Each item carries the same fields:

```text
item_id: stable identifier, so results can cite their evidence
title: short human-readable name
content: the approved teaching content itself
tags: labels the matching step can score against
risk_level: low / medium / high, a hook for stricter handling later
```

In a production system, the equivalent data might come from public documentation, approved knowledge bases, model cards, dataset cards, or authorised internal systems. This practical does not touch those live sources — but the workflow shape around the data would stay the same if it did.

In [ ]:
# LOCAL_ITEMS is the entire "world" the mandatory workflow is allowed to know.
# Here it stands in for an approved provider fact sheet: the latency, cost and
# throughput notes an engineering team keeps before running a real benchmark.
# Keeping the collection this small is a teaching choice: you can hold all of
# it in your head, so any surprising output must come from the code, not the data.
LOCAL_ITEMS = [
    {
        "item_id": "M07E-001",
        "title": "Latency Basics",
        "content": "This item explains latency in the context of Fast Inference Provider Comparison. It is approved synthetic teaching content.",
        "tags": ["latency", "basics", "approved"],
        "risk_level": "low"
    },
    {
        "item_id": "M07E-002",
        "title": "Cost Practice",
        "content": "This item describes how cost can be handled through validation, inspection and structured output.",
        "tags": ["cost", "practice", "validation"],
        "risk_level": "low"
    },
    {
        "item_id": "M07E-003",
        "title": "Throughput Safety",
        "content": "This item highlights the safety boundary for throughput and explains why unsupported claims or external side effects should be avoided.",
        "tags": ["throughput", "safety", "boundary"],
        "risk_level": "medium"
    },
]

print("Number of local items:", len(LOCAL_ITEMS))
print(json.dumps(LOCAL_ITEMS[0], indent=2))

The local items play the same role as a small approved knowledge base, state table, model-card list, evaluation table, or policy scenario list. The purpose is not to cover every real-world case. The purpose is to make the workflow observable: with only three items, you can always explain exactly why a result was — or was not — produced. If you later replace this list with a real data source, everything around it (validation, selection, inspection, tests) should keep working unchanged. That clean separation between data and control logic is itself one of the design lessons of this module.

<a id="m07e-workflow"></a>

### 5. Mandatory Local Workflow

The workflow is built from four small functions, each doing one job:

```text
request (a plain string)
   |
   v
validate_request ........ is the request non-empty and inside the
   |                      safety boundary? If not: stop early.
   v
select_relevant_items ... which approved items overlap with the
   |                      request? Keep the best top_k matches.
   v
build_structured_result . turn the selected evidence into a result
   |                      with status, summary and limitations.
   v
run_local_workflow ...... the orchestrator that chains the three
                          steps and stops at the first problem.
```

Every function returns the same envelope shape — `{"ok": ..., "error": ..., "result": ...}` — so the caller always checks success in the same way. This mirrors how agent frameworks wrap tool calls: a predictable envelope makes error handling boring, and boring error handling is reliable error handling.

The implementation is intentionally explicit rather than compact. In teaching notebooks, readable logic is more valuable than clever one-line code.

In [ ]:
def normalise_text(text: str) -> str:
    # Collapse whitespace and lowercase, so matching is not fooled by
    # formatting differences ("Fine-Tuning" vs "fine-tuning").
    return re.sub(r"\s+", " ", text.lower()).strip()


def tokenise(text: str) -> List[str]:
    # Non-string input yields an empty token list instead of raising:
    # the workflow prefers a controlled "no match" over a crash.
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z_]+", normalise_text(text))


def validate_request(request: str) -> Dict[str, Any]:
    # Gate 1 (failure case): input we cannot even process.
    if not isinstance(request, str) or not request.strip():
        return {"ok": False, "error": "request must be a non-empty string.", "result": None}

    # Gate 2 (refusal case): input we can process but must not serve.
    # A simple term denylist stands in for the safety classifiers a real
    # system would use; the design point is WHERE the check sits (before
    # any other work happens), not how sophisticated it is.
    lower = request.lower()
    unsafe_terms = [
        "private file", "password", "api key", "credential", "send email",
        "delete all", "shell command", "student record", "hidden solution"
    ]

    if any(term in lower for term in unsafe_terms):
        # Note ok=True here. Refusing is the workflow WORKING correctly,
        # not an internal error -- so the envelope reports success with
        # allowed=False, and the caller turns that into a refusal result.
        return {
            "ok": True,
            "error": None,
            "result": {
                "allowed": False,
                "reason": "The request asks for private data, credentials, hidden material or external side effects."
            }
        }

    return {
        "ok": True,
        "error": None,
        "result": {
            "allowed": True,
            "reason": "The request is allowed for the local teaching workflow."
        }
    }

In [ ]:
def select_relevant_items(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # Guard the parameter students most often change. Rejecting bad values
    # up front beats producing confusing half-results later.
    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    # Relevance = how many words the request shares with an item's title,
    # content and tags. This bag-of-words overlap is a deliberately
    # transparent stand-in for the embedding similarity a production
    # pipeline would use: you can verify every score by hand, which you
    # cannot do with embeddings.
    request_terms = set(tokenise(request))
    scored = []

    for item in items:
        item_text = " ".join([
            item.get("title", ""),
            item.get("content", ""),
            " ".join(item.get("tags", [])),
        ])
        item_terms = set(tokenise(item_text))
        score = len(request_terms.intersection(item_terms))
        # Items with zero overlap are dropped entirely. This is the
        # anti-hallucination rule: an item with no connection to the
        # request must never be presented as supporting evidence.
        if score > 0:
            selected = dict(item)  # copy, so scoring never mutates LOCAL_ITEMS
            selected["score"] = score
            scored.append(selected)

    # Highest overlap first; keep only the strongest top_k matches so the
    # result stays small enough to inspect by eye.
    scored.sort(key=lambda item: item["score"], reverse=True)

    return {"ok": True, "error": None, "result": scored[:top_k]}

In [ ]:
def build_structured_result(request: str, selected_items: List[Dict[str, Any]]) -> Dict[str, Any]:
    # Edge case first: no relevant evidence. The honest answer is
    # "insufficient context", never an invented one. Making this an
    # explicit, testable status is what separates a controlled workflow
    # from a model that fills gaps with plausible-sounding text.
    if not selected_items:
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "insufficient_context",
                "summary": "The approved local data does not contain enough information to complete this request.",
                "selected_items": [],
                "limitations": [
                    "No sufficiently relevant local item was selected.",
                    "The workflow should not invent missing information."
                ]
            }
        }

    # Normal case: summarise ONLY what the selected evidence supports,
    # and say so explicitly in the summary text.
    summary = (
        "The workflow selected approved local items and produced a structured result for "
        "Fast Inference Provider Comparison. The result is based only on selected local evidence."
    )

    # Limitations are part of every successful result, not an apology added
    # when something breaks. Readers should always be told the boundaries
    # of what they are looking at.
    return {
        "ok": True,
        "error": None,
        "result": {
            "status": "completed",
            "summary": summary,
            "selected_items": selected_items,
            "limitations": [
                "This is a local teaching workflow, not a live external system.",
                "The result should be checked before being reused in a real setting."
            ]
        }
    }

In [ ]:
def run_local_workflow(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # Orchestrator pattern: run each stage in order, stop at the first
    # problem, and pass structured envelopes (never raw exceptions) upward.

    validation = validate_request(request)
    if not validation["ok"]:
        return validation  # invalid input: propagate the error envelope unchanged

    if not validation["result"]["allowed"]:
        # A refusal is a first-class outcome with its own status, so tests
        # can assert on it exactly like they assert on a completed result.
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "refused",
                "summary": validation["result"]["reason"],
                "selected_items": [],
                "limitations": ["The request is outside the allowed safety boundary."]
            }
        }

    selected = select_relevant_items(request, items, top_k=top_k)
    if not selected["ok"]:
        return selected

    structured = build_structured_result(request, selected["result"])
    if not structured["ok"]:
        return structured

    # Echo the request back into the result so the output is
    # self-describing: anyone reading it later knows what was asked.
    return {
        "ok": True,
        "error": None,
        "result": {
            "request": request,
            **structured["result"]
        }
    }


# Smoke test: one allowed request, end to end. The words "validation" and
# "safety" overlap with the local items, so this should complete.
example_result = run_local_workflow("Explain validation and safety boundary", LOCAL_ITEMS)
example_result

<a id="m07e-inspection"></a>

### 6. Inspection and Interpretation

The output above should be inspected rather than accepted blindly. This is the same evaluation habit you have practised across M05–M08, and it applies with extra force in this module: a benchmark table can look authoritative while hiding cold-start outliers or mismatched settings, so you must always check how the numbers were produced.

Work through these questions against the result you just produced:

```text
1. Was the request allowed?
2. Which local items were selected, and with what scores?
3. Are the selected items actually relevant to the request?
4. Did the workflow state its limitations?
5. Would it have refused an unsafe version of the same request?
```

In [ ]:
def display_workflow_result(result: Dict[str, Any]) -> None:
    # A human-readable view of the result envelope. Printing a curated
    # summary (instead of dumping raw JSON) makes classroom inspection
    # faster: status first, then evidence, then limitations.
    if not result.get("ok"):
        print("ERROR:", result.get("error"))
        return

    payload = result["result"]
    print("Status:", payload.get("status"))
    print("Summary:", payload.get("summary"))

    print("\nSelected items:")
    if not payload.get("selected_items"):
        print("- None")
    for item in payload.get("selected_items", []):
        # The score explains WHY this item was chosen -- always show it.
        print(f"- {item['item_id']} | score={item.get('score')} | {item['title']}")
        print(f"  {item['content']}")

    print("\nLimitations:")
    for limitation in payload.get("limitations", []):
        print("-", limitation)


display_workflow_result(example_result)

A strong result is not necessarily the longest result. A strong result is inspectable, grounded in the selected items, and clear about its limitations. If a selected item is irrelevant, the final result should not be trusted — no matter how confident the summary sounds. Carry this standard forward: apply exactly the same checks to your own extension in the Student Tasks, and to anything you run in the optional section.

<a id="m07e-optional"></a>

### 7. Optional Real Model or Package Section

This section is optional. The mandatory workflow above already demonstrates the design pattern; this section explains how the same skeleton would host live latency and cost measurements against real providers. If you do not have the required access, skipping is a perfectly acceptable outcome — record it exactly as shown at the end of this section.

The correct integration pattern keeps every safety stage and swaps only the middle:

```text
Validated request
      |
      v
Selected approved context
      |
      v
Real step: timed provider calls   <-- the ONLY stage that changes
      |
      v
Structured result
      |
      v
Inspection, limitations, tests
```

A real version of this session would time actual providers, using keys you manage through the `getpass` pattern:

- **Method.** Use one fixed prompt set and identical `max_tokens` across providers. Send one warm-up request first (excluded from results), then take at least 3 timed measurements per provider with `time.perf_counter()` and report the median, not the mean — a single cold start can double a mean.
- **What to record.** Time to first token, total time, output tokens per second, cost from the provider's published per-token prices, and the date, because prices and speeds change.
- **What you should observe.** Providers rank differently on different columns, and the "winner" changes when you change the weights — which is exactly the lesson of the scoring matrix.
- **Failure behaviour.** HTTP 429 means you hit a rate limit — add a delay between calls and retry with backoff. Wildly inconsistent timings usually mean you forgot the warm-up call or are comparing different output lengths.

Whatever you run: never hard-code API keys (use the `getpass` pattern from earlier modules) and never use private data. If the optional section is not available to you, write:

```text
Skipped: optional package/API access not available.
```

In [ ]:
# Optional package/API section.
# This placeholder is intentionally safe: it makes no external calls, and
# the mandatory part of the session does not depend on anything below.

def optional_external_version_available() -> bool:
    # Flip this to True only if you have the access described above and
    # have read the sketch below. Defaulting to False keeps the notebook
    # runnable end-to-end for everyone.
    return False

if not optional_external_version_available():
    print("Skipped: optional package/API access not available.")

# --- Sketch of the real integration (commented on purpose; adapt before running) ---
# 1) Load keys safely -- never paste them into a cell:
#      import os
#      from getpass import getpass
#      os.environ["PROVIDER_API_KEY"] = getpass("Enter PROVIDER_API_KEY: ")
# 2) Use ONE fixed prompt set and identical max_tokens for every provider.
# 3) Send a warm-up request first and EXCLUDE it from your results.
# 4) Time each call with time.perf_counter(); repeat >= 3 times per
#    provider and report the MEDIAN (a cold start can double a mean).
# 5) Record the date next to every cost figure -- provider prices change.
# 6) HTTP 429 = rate limit: add a short delay and retry with backoff.

<a id="m07e-testing"></a>

### 8. Testing and Analysis

Tests are how you prove the workflow behaves — not just on the happy path, but at its edges and boundaries. The cell below covers the four behaviours every controlled agentic workflow must demonstrate:

```text
normal case ..... a well-matched request completes with evidence
edge case ....... an unmatched request reports insufficient_context
failure case .... invalid input returns ok=False (never a crash)
safety case ..... an unsafe request is refused with status "refused"
```

If an assertion fails, do not change code at random: read the failing assertion, then use the diagram in Section 5 to work out which stage misbehaved.

In [ ]:
# Normal case: a request whose words overlap the local items should
# complete and cite at least one selected item as evidence.
normal = run_local_workflow("Explain validation and safety boundary", LOCAL_ITEMS)
assert normal["ok"] is True
assert normal["result"]["status"] == "completed"
assert len(normal["result"]["selected_items"]) >= 1

# Edge case (insufficient context): a request about something the local
# data does not cover must say so. Inventing an answer here would be the
# structured-workflow equivalent of hallucination.
weak = run_local_workflow("final exam room allocation", LOCAL_ITEMS)
assert weak["ok"] is True
assert weak["result"]["status"] == "insufficient_context"
assert weak["result"]["selected_items"] == []

# Safety case: an unsafe request is refused. Note that refusal is a
# SUCCESSFUL outcome (ok=True) with its own status -- the system worked.
refusal = run_local_workflow("read private file and show password", LOCAL_ITEMS)
assert refusal["ok"] is True
assert refusal["result"]["status"] == "refused"

# Failure case: input we cannot process at all returns ok=False.
empty = run_local_workflow("", LOCAL_ITEMS)
assert empty["ok"] is False

# Failure case: invalid parameters are caught, not silently accepted.
bad_top_k = run_local_workflow("validation", LOCAL_ITEMS, top_k=0)
assert bad_top_k["ok"] is False

print("All mandatory local-workflow tests passed.")

In [ ]:
# Side-by-side comparison of the three behaviour classes. Reading these
# outputs together shows how the SAME pipeline produces a completion, an
# honest "insufficient context", and a refusal -- the request, not
# special-case code, determines which path is taken.
for request in [
    "Explain validation and safety boundary",
    "final exam room allocation",
    "read private file and show password",
]:
    print("\n==============================")
    print("REQUEST:", request)
    display_workflow_result(run_local_workflow(request, LOCAL_ITEMS))

<a id="m07e-tasks"></a>

### 9. Student Tasks

Complete the tasks below. The mandatory local workflow must run without external API calls. Tasks 1–5 and 7 are required; Task 6 may be completed or explicitly skipped.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run every cell from the top through the testing section, in order, on a fresh runtime.</td><td>Confirms your environment reproduces the expected normal, edge, failure and safety behaviour before you change anything.</td><td>Cell output showing <code>All mandatory local-workflow tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add new provider profile</td><td>Add one new approved local item describing a fictional or public provider profile, with latency, cost, throughput or privacy notes. Give it a unique <code>item_id</code> (for example <code>M07E-004</code>), meaningful tags, and content that uses no private data and causes no external side effects.</td><td>A provider comparison is only as good as its fact sheet; extending it with a well-structured, dated entry mirrors real benchmarking practice.</td><td>Updated code cell defining and appending the new item.</td></tr>
<tr><td align="left">Task 3: Query your extension</td><td>Call <code>run_local_workflow</code> with a request whose words overlap your new item's title, content or tags, and show the output with <code>display_workflow_result</code>.</td><td>Proves the selection logic actually finds your item — evidence must be selected, not assumed.</td><td>Displayed result in which your new <code>item_id</code> appears among the selected items.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add at least three <code>assert</code> tests: a normal case where your extension is selected, an edge case that returns <code>insufficient_context</code>, and a failure or refusal case (invalid input, or an unsafe request).</td><td>Normal, edge and failure coverage is the minimum standard for executable teaching material — and for production agents.</td><td>A test cell that runs without raising and covers all three behaviours.</td></tr>
<tr><td align="left">Task 5: Analyse grounding</td><td>Write a short paragraph identifying which selected item supports the Task 3 result, and whether any claim in the summary is unsupported by that evidence.</td><td>Grounding analysis is the habit that catches hallucination — in this local workflow, and in any benchmark table you publish to your team.</td><td>A markdown cell of 4–8 sentences.</td></tr>
<tr><td align="left">Task 6: Optional section</td><td>Run the optional section safely if you have access (provider API keys); otherwise write <code>Skipped: optional package/API access not available.</code></td><td>Practises the discipline of recording what was and was not executed — an honest skip beats a broken attempt.</td><td>Optional-section output, or the skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Write 150–250 words on what this workflow teaches about provider selection and agentic AI design.</td><td>Articulating the design lesson is how the pattern transfers to your own future projects.</td><td>A markdown cell of 150–250 words.</td></tr>
</tbody>
</table>

</div>

For Task 4, the three behaviours mean: **normal** — your extension is selected and the status is `completed`; **edge** — a genuinely unrelated request returns `insufficient_context` with no selected items; **failure** — invalid input returns `ok=False`, or an unsafe request returns status `refused`.

In [ ]:
# Student task starter for M07E (Tasks 2 and 3).
#
# How to use this starter:
#   1. Copy the example below into live code (remove the leading '#').
#   2. Change item_id, title, content and tags to YOUR extension idea
#      (see Task 2 in the table above). Keep it synthetic and approved:
#      no private data, no credentials, no external side effects.
#   3. Append the item to LOCAL_ITEMS, then query it with a request that
#      shares whole words with your title, content or tags -- selection
#      is based on word overlap, so fragments will not match.
#   4. Show the result with display_workflow_result and check that your
#      item_id appears among the selected items.
#
# Example:
# new_item = {
#     "item_id": "M07E-004",
#     "title": "Human Review Extension",
#     "content": "Human review is important before outputs from Fast Inference Provider Comparison are used in real settings.",
#     "tags": ["human_review", "safety", "extension"],
#     "risk_level": "low"
# }
#
# LOCAL_ITEMS.append(new_item)
# result = run_local_workflow("Why is human review important?", LOCAL_ITEMS)
# display_workflow_result(result)

<a id="m07e-submission"></a>

### 10. Submission and Reflection

Submit the completed notebook with all outputs visible. Required items:

```text
1. Mandatory baseline test output ("All mandatory local-workflow tests passed.").
2. Your extension code (the new approved item or rule).
3. Workflow output showing your extension was actually selected.
4. At least three added tests using assert statements (normal, edge, failure/refusal).
5. Short grounding or support analysis (4-8 sentences).
6. Optional package/API result, or the explicit skipped note.
7. 150-250 word reflection.
```

**Quality checks before you submit.** Restart the runtime and run all cells top to bottom (in Colab: `Runtime` > `Restart session and run all`); every cell must execute without errors, and the outputs you submit must come from that clean run. Check that your extension item has a unique `item_id`, that no cell contains a real API key or private data, and that your added tests would genuinely fail if the behaviour they check were broken — try breaking one on purpose, watch it fail, then fix it.

**Debugging guide.** The most common problems in this notebook, and where to look:

- `NameError` (for example `run_local_workflow is not defined`): cells were run out of order — restart and run all cells from the top.
- Your new item is never selected: the request shares no words with the item's title, content or tags. `tokenise` only extracts letter/underscore words, so requests must share whole words with the item, not fragments.
- A test expecting `completed` gets `insufficient_context`: the request's word overlap with every item scored zero — reword the request, or enrich the item's tags.
- A test expecting `insufficient_context` gets `completed`: your edge-case request accidentally shares words with an item — pick genuinely unrelated wording.
- An unsafe request is not refused: the denylist in `validate_request` matches exact phrases — check the spelling, or extend the list as part of your extension.
- (Optional section) HTTP 429 or wildly inconsistent timings: you hit a rate limit or skipped the warm-up call — space out requests, retry with backoff, and report medians rather than means.

Reflection questions to address in your 150–250 words:

1. What are the main stages of the workflow, and what does each stage protect against?
2. Why does the workflow validate input before doing any other work?
3. What should happen when there is insufficient approved context, and why is that better than answering anyway?
4. Why should unsafe requests be refused explicitly rather than quietly ignored?
5. How does this session connect to later agentic AI systems, where provider choice affects the latency and cost of every tool call an agent makes?

#### Further Readings

- [Text Generation Inference documentation](https://huggingface.co/docs/text-generation-inference/index)
- [vLLM documentation](https://docs.vllm.ai/)
- [MLPerf Inference benchmark results](https://mlcommons.org/benchmarks/inference-datacenter/)